# Connect to Neo4j — deluxe-analyze

Replace `NEO4J_IP` with the static IP from `terraform output neo4j_static_ip`.
Add your Colab/notebook IP to `allowed_neo4j_ips` in `infra/terraform.tfvars` and re-apply.

In [ ]:
import os
from neo4j import GraphDatabase

NEO4J_IP = os.environ.get("NEO4J_IP", "REPLACE_ME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "REPLACE_ME")

driver = GraphDatabase.driver(
    f"bolt://{NEO4J_IP}:7687",
    auth=("neo4j", NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Connected to Neo4j")

In [ ]:
with driver.session(database="neo4j") as session:
    # Basic connectivity check
    result = session.run("MATCH (n) RETURN count(n) AS total")
    print(f"Total nodes: {result.single()['total']}")

    # Degree centrality via GDS
    result = session.run("""
        CALL gds.degree.stream('social-graph')
        YIELD nodeId, score
        RETURN gds.util.asNode(nodeId).id AS user_id, score AS degree
        ORDER BY degree DESC
        LIMIT 10
    """)
    for row in result:
        print(f"  {row['user_id']}: degree={row['degree']}")

In [ ]:
# Crear proyeccion del grafo social
with driver.session(database="neo4j") as session:
    # Proyeccion en memoria
    session.run("""
        CALL gds.graph.project(
            'social-graph',
            'Usuario',
            {
                CONOCE_A: {
                    orientation: 'UNDIRECTED',
                    properties: ['tie_strength']
                }
            }
        )
    """)

    # Louvain community detection
    result = session.run("""
        CALL gds.louvain.stream('social-graph')
        YIELD nodeId, communityId
        RETURN communityId, count(*) AS members
        ORDER BY members DESC
        LIMIT 10
    """)

    print("Top 10 comunidades (Louvain):")
    for row in result:
        print(f"  comunidad {row['communityId']}: {row['members']} miembros")

In [ ]:
# Degree centrality — usuarios mas conectados
with driver.session(database="neo4j") as session:
    result = session.run("""
        CALL gds.degree.stream('social-graph')
        YIELD nodeId, score
        MATCH (u:Usuario) WHERE id(u) = nodeId
        RETURN u.id AS user_id, score AS degree
        ORDER BY degree DESC
        LIMIT 10
    """)

    print("Top 10 usuarios por grado (CONOCE_A):")
    for row in result:
        print(f"  {row['user_id']}: {int(row['degree'])} conexiones")

## Nota: IPs rotativas en Colab

Colab usa IPs rotativas que cambian entre sesiones. Opciones:

**Opcion A — Anadir rango Colab a whitelist** (menos seguro):
Anadir  a  en  y re-aplicar.

**Opcion B — IAP tunnel** (recomendado para acceso local):

Luego conectar desde un notebook local a .